# CycleGAN 数据预处理：CelebA 黑发 ⇄ 金发

数据逻辑全部封装在同目录的 `gan_data.py` 里，本 notebook 只负责调用。

| 函数 | 作用 |
| --- | --- |
| `prepare_celeba()` | 一次性：把 CelebA 拆成 `trainA` / `trainB` / `testA` / `testB`（拷贝，原图不动） |
| `get_dataloaders()` | 返回 `CycleGANData(train_A, train_B, test_A, test_B)` 四个 DataLoader |
| `unpaired(a, b)` | 非配对取样，一轮产出 `len(a)` 对 `(real_A, real_B)`，取完即停 |
| `denorm(t)` | 把 `[-1, 1]` 的张量还原到 `[0, 1]`，方便 imshow |
| `build_transform(size, train)` | 单独取变换（想自己写 Dataset 时用） |
| `FlatImageDataset(root)` | 扁平目录 Dataset（`ImageFolder` 用不了，原因见下） |

## 运行前必须准备

1. **原始图片** —— `data/img_align_celeba/img_align_celeba/`，202599 张 jpg（已就位）。
2. **属性表** —— `data/list_attr_celeba.csv`（已就位）。

## 产出

```
data/
├── trainA/  38775 张  黑发训练集
├── testA/    9694 张  黑发测试集
├── trainB/  23984 张  金发训练集
└── testB/    5996 张  金发测试集
```

## 三个设计要点

- **拷贝而不是移动**：原始 202599 张图全程不动，脚本可反复运行、中途报错也不用重下数据集。
- **丢弃双标签**：CelebA 有 3 张图 `Black_Hair` 与 `Blond_Hair` 同时为 1，放哪个域都会给 CycleGAN 引入错误配对，直接丢弃。
- **为什么不用 `ImageFolder`**：它要求目录下有一层类别子文件夹，而 `data/trainA` 是扁平的，在 torchvision 0.26 上会报 `FileNotFoundError: Couldn't find any class folder in ...`。所以用 `FlatImageDataset`。

> 想先快速跑通训练：把下面改成 `prepare_celeba(max_per_domain=2000)`。


In [ ]:
import sys

import matplotlib.pyplot as plt

sys.path.append(r'C:\Users\moneyforever\Desktop\Deep-Learning\GANs\CycleGAN')
from gan_data import ROOT, denorm, get_dataloaders, prepare_celeba, unpaired

In [ ]:
# 一次性：把 CelebA 拆成 trainA/trainB/testA/testB（拷贝，原图不动）。
# 已经拆好就跳过 —— 重新运行会清空并重拷 78449 张，没有必要。
DOMAINS = ('trainA', 'testA', 'trainB', 'testB')

_ready = all(
    (ROOT / 'data' / d).is_dir() and any((ROOT / 'data' / d).glob('*.jpg'))
    for d in DOMAINS
)

if _ready:
    print('数据已就绪，跳过拆分。要强制重建请直接调用 prepare_celeba()')
    stats = {d: len(list((ROOT / 'data' / d).glob('*.jpg'))) for d in DOMAINS}
else:
    stats = prepare_celeba()
stats

In [ ]:
# 训练时反复调用这里即可
dl = get_dataloaders(size=256, batch_size=1, num_workers=0)

print('A 域训练集 %6d 张 -> %d 个 batch' % (len(dl.train_A.dataset), len(dl.train_A)))
print('B 域训练集 %6d 张 -> %d 个 batch' % (len(dl.train_B.dataset), len(dl.train_B)))
print('A 域测试集 %6d 张' % len(dl.test_A.dataset))
print('B 域测试集 %6d 张' % len(dl.test_B.dataset))

# 非配对取样：A、B 各自独立打乱，数量不等也不会被截断
real_A, real_B = next(iter(unpaired(dl.train_A, dl.train_B)))
print()
print('real_A', tuple(real_A.shape), real_A.dtype,
      'range [%.2f, %.2f]' % (real_A.min(), real_A.max()))
print('real_B', tuple(real_B.shape), real_B.dtype,
      'range [%.2f, %.2f]' % (real_B.min(), real_B.max()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, t, title in ((axes[0], real_A, 'real_A (Black_Hair)'),
                     (axes[1], real_B, 'real_B (Blond_Hair)')):
    ax.imshow(denorm(t[0]).permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()